# Attention Rollout & Flow

Individual attention patterns only tell part of the story — information
flows through **multiple layers**. Attention rollout chains attention
matrices across layers to compute the effective attention from output
to input. Attention flow additionally accounts for the residual stream.

This notebook covers the `irtk.attention_rollout` module.

## Why This Matters

Attention rollout aggregates attention across layers to estimate the total information flow from input tokens to output positions. Unlike single-layer patterns, rollout accounts for the compositional nature of multi-layer attention.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import attention_rollout

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Layer-Aggregated Attention

First, combine attention across heads within a single layer.

In [ ]:
prompt = "The Eiffel Tower is in"
tokens = model.to_tokens(prompt)
token_strs = [tokenizer.decode([int(t)]) for t in tokens]
_, cache = model.run_with_cache(tokens)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, layer in enumerate([0, 3, 6, 8, 10, 11]):
    ax = axes[idx // 3][idx % 3]
    agg = attention_rollout.layer_aggregated_attention(
        cache, layer=layer, n_heads=model.cfg.n_heads
    )
    im = ax.imshow(agg, cmap='Blues', aspect='auto', vmin=0)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(token_strs, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(token_strs, fontsize=8)
    ax.set_title(f'Layer {layer}')

plt.suptitle('Mean Attention per Layer', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Effective Attention

The residual stream carries information unchanged through layers.
Effective attention mixes the attention pattern with the identity
to reflect this.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, w in zip(axes, [0.0, 0.5, 1.0]):
    eff = attention_rollout.effective_attention(
        cache, layer=6, n_heads=model.cfg.n_heads, residual_weight=w
    )
    im = ax.imshow(eff, cmap='Blues', aspect='auto', vmin=0)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(token_strs, rotation=45, ha='right')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(token_strs)
    ax.set_title(f'Layer 6, residual_weight={w}')
    plt.colorbar(im, ax=ax)

plt.suptitle('Effective Attention: Identity ↔ Attention mixing', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Attention Rollout

Chain attention matrices across layers: R = A_L @ A_{L-1} @ ... @ A_1.
This gives the effective attention from each output position to each input.

In [ ]:
rollout = attention_rollout.attention_rollout(model, tokens)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full rollout matrix
im = axes[0].imshow(rollout, cmap='Blues', aspect='auto', vmin=0)
axes[0].set_xticks(range(len(tokens)))
axes[0].set_xticklabels(token_strs, rotation=45, ha='right')
axes[0].set_yticks(range(len(tokens)))
axes[0].set_yticklabels(token_strs)
axes[0].set_xlabel('Input')
axes[0].set_ylabel('Output')
axes[0].set_title('Attention Rollout')
plt.colorbar(im, ax=axes[0])

# Last position attribution
attr = rollout[-1]
axes[1].bar(range(len(tokens)), attr, color='steelblue')
axes[1].set_xticks(range(len(tokens)))
axes[1].set_xticklabels(token_strs, rotation=45, ha='right')
axes[1].set_ylabel('Rollout weight')
axes[1].set_title('Input Attribution for Last Position')

plt.tight_layout()
plt.show()

print("Last position rollout attribution:")
for i, (tok, w) in enumerate(zip(token_strs, attr)):
    bar = '█' * int(w * 50)
    print(f"  {tok:>10s}: {w:.4f} {bar}")

## 4. Attention Flow

Like rollout but accounts for the residual stream: each layer
mixes attention with the identity. This is often more accurate.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, w in zip(axes, [0.3, 0.5, 0.8]):
    flow = attention_rollout.attention_flow(model, tokens, residual_weight=w)
    attr = flow[-1]
    ax.bar(range(len(tokens)), attr, color='coral')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(token_strs, rotation=45, ha='right')
    ax.set_ylabel('Flow weight')
    ax.set_title(f'Attention Flow (w={w})')

plt.suptitle('Effect of Residual Weight on Attention Flow', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Rollout vs Flow Comparison

Compare plain rollout with residual-aware flow attribution.

In [ ]:
rollout_attr = attention_rollout.token_attribution_rollout(model, tokens, method="rollout")
flow_attr = attention_rollout.token_attribution_rollout(
    model, tokens, method="flow", residual_weight=0.5
)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(tokens))
ax.bar(x - 0.2, rollout_attr, 0.4, label='Rollout', color='steelblue')
ax.bar(x + 0.2, flow_attr, 0.4, label='Flow (w=0.5)', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_ylabel('Attribution weight')
ax.set_title('Rollout vs Flow Attribution')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Per-Head Rollout

Isolate a single head's contribution to the overall attention flow
by using its pattern at its layer while averaging all other layers.

In [ ]:
# Compare a few heads
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for idx, (layer, head) in enumerate([(0, 0), (3, 5), (5, 1), (9, 6), (9, 9), (11, 10)]):
    ax = axes[idx // 3][idx % 3]
    attr = attention_rollout.per_head_rollout(model, tokens, layer=layer, head=head)
    ax.bar(range(len(tokens)), attr, color='steelblue')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(token_strs, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Attribution')
    ax.set_title(f'L{layer}H{head}')

plt.suptitle('Per-Head Rollout Attribution', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `layer_aggregated_attention()` | Combine heads within a layer (mean/max) |
| `effective_attention()` | Mix attention with identity (residual) |
| `attention_rollout()` | Chain attention across layers |
| `attention_flow()` | Rollout with residual stream contribution |
| `token_attribution_rollout()` | Per-input attribution for an output position |
| `per_head_rollout()` | Isolate one head's contribution to rollout |